# Demostracion del Orquestador Jerarquico Multi-Agente con LangGraph

Este Jupyter Notebook forma parte del proyecto **Multi-Agent Orchestrator** para Coderhouse.

## Objetivos de la Demostracion
1. **Demostrar la Arquitectura Jerarquica**: Ver como el nodo **Supervisor** centraliza las decisiones de enrutamiento con Salida Estructurada hacia el **Investigador** y el **Analista**.
2. **Prevencion del 'Supervisor Infinito'**: Validacion de condiciones de suficiencia de datos (`research_data`, `analysis_data`) y control estricto de iteraciones maximas (`step_count >= 6`).
3. **Uso de Herramientas Reales**: Validacion del Agente de Busqueda y del Agente Analista con herramientas de computo.
4. **Resiliencia & Ejecucion Multiproveedor**: Demostracion de soporte para Gemini, OpenAI, Groq y degradacion transparente a `MockLLM` en entornos sin credenciales.

In [ ]:
# 1. Importacion de modulos principales del proyecto
import asyncio
from langchain_core.messages import HumanMessage
from app.graph import build_graph
from app.state import TeamState
from app.config import get_llm

# Comprobamos el proveedor activo
llm = get_llm()
print(f"Proveedor de LLM inicializado: {llm.__class__.__name__}")

# Construimos el grafo (sin checkpointer para pruebas locales en el notebook)
graph = build_graph()

## Ejecucion y Streaming de Transiciones del Grafo Jerarquico
Visualizamos en directo la ejecucion del orquestador mediante el metodo asincrono `.astream()`.

In [ ]:
# Definicion del estado inicial de prueba
initial_state = {
    "messages": [HumanMessage(content="Investigar la adopcion de LangGraph en 2026 y calcular metricas de impacto.")],
    "next_agent": "Supervisor",
    "research_data": None,
    "analysis_data": None,
    "step_count": 0,
    "is_critical": False,
    "approved_by_human": False
}

print("=" * 80)
print("EJECUTANDO ORQUESTADOR JERARQUICO MULTI-AGENTE")
print("=" * 80)

async def run_graph():
    # Streaming asincrono del flujo de ejecucion nodo a nodo
    async for step_output in graph.astream(initial_state):
        for node_name, state_update in step_output.items():
            print(f"\n[NODO ALCANZADO]: {node_name}")
            if "next_agent" in state_update:
                print(f"   -> Transicion decidida: {state_update['next_agent']}")
            if "research_data" in state_update and state_update["research_data"]:
                print(f"   Resultado Investigacion: {state_update['research_data'][:120]}...")
            if "analysis_data" in state_update and state_update["analysis_data"]:
                print(f"   Resultado Analisis: {state_update['analysis_data'][:120]}...")
            if "step_count" in state_update:
                print(f"   Contador de pasos: {state_update['step_count']}")

    print("\n" + "=" * 80)
    print("FLUJO FINALIZADO CORRECTAMENTE SIN BUCLES INFINITOS")
    print("=" * 80)

# Ejecutamos la funcion asincrona (requerido por Jupyter/Colab)
await run_graph()